# 第 36 章：Agentic Research Assistants、工具调用与验证边界

这个 notebook 用一个极小的瞬变候选体分诊表，演示 agentic workflow 和普通“一次性回答”之间的关键差别：

- baseline 只按一个分数排序，很容易把 artifact 和 moving object 推到前面
- tool-using agent 会先做显式过滤、再做分步评分、最后把边界样本送入人工复核
- 真正可靠的关键不在于“代理感”本身，而在于工具边界、停止条件和 handoff 规则

教学说明：这里不依赖任何在线模型或 agent 框架；重点是把 multi-step research workflow 的结构讲清楚。


In [ ]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path

DATA_PATH = Path("../../data/small/transient_agent_workflow_demo.csv").resolve()

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    row["alert_score"] = float(row["alert_score"])
    row["rise_rate_mag_day"] = float(row["rise_rate_mag_day"])
    row["color_g_r"] = float(row["color_g_r"])
    row["crossmatch_arcsec"] = float(row["crossmatch_arcsec"])

print(f"Loaded {len(rows)} transient candidates from {DATA_PATH.name}")
print("image-quality counts:", dict(Counter(row["image_quality"] for row in rows)))
print("moving-object counts:", dict(Counter(row["moving_object_flag"] for row in rows)))
print("reference actions:", dict(Counter(row["reference_action"] for row in rows)))
print("First five rows:")
for row in rows[:5]:
    print(row)


In [ ]:
baseline_ranked = sorted(rows, key=lambda row: row["alert_score"], reverse=True)

print("Single-shot baseline: rank only by alert_score")
for row in baseline_ranked[:5]:
    print(row["candidate_id"], row["alert_score"], row["image_quality"], row["moving_object_flag"], row["reference_action"])

baseline_top3 = baseline_ranked[:3]
baseline_precision_at_3 = sum(row["reference_action"] == "follow_now" for row in baseline_top3) / len(baseline_top3)
print("baseline top-3 ids:", [row["candidate_id"] for row in baseline_top3])
print("baseline precision@3:", round(baseline_precision_at_3, 3))


In [ ]:
def novelty_flag(row):
    return row["crossmatch_arcsec"] > 1.5


def blue_candidate_flag(row):
    return row["color_g_r"] < 0.1


def followup_score(row):
    novelty = 1.0 if novelty_flag(row) else 0.0
    blue_bonus = 1.0 if blue_candidate_flag(row) else 0.0
    rise_component = row["rise_rate_mag_day"] / 0.45
    return 0.50 * row["alert_score"] + 0.25 * rise_component + 0.15 * novelty + 0.10 * blue_bonus


def route_candidate(row):
    score = followup_score(row)

    if row["image_quality"] == "artifact":
        return score, "reject", "artifact gate"
    if row["moving_object_flag"] == "yes":
        return score, "reject", "known moving-object gate"
    if row["crossmatch_arcsec"] <= 0.8:
        return score, "reject", "too close to known source"
    if row["image_quality"] != "clean":
        return score, "manual_review", "non-clean image quality"
    if row["crossmatch_arcsec"] <= 1.5:
        return score, "manual_review", "weak novelty support"
    if 0.58 <= score < 0.78:
        return score, "manual_review", "borderline follow-up score"
    if score >= 0.78:
        return score, "follow_now", "clear high-priority candidate"
    return score, "reject", "score too low"


def run_agentic_triage(candidates):
    action_log = []
    decisions = []
    for row in candidates:
        score, action, rationale = route_candidate(row)
        action_log.append({
            "candidate_id": row["candidate_id"],
            "tool_sequence": [
                "inspect_schema",
                "artifact_gate",
                "moving_object_gate",
                "novelty_check",
                "priority_score",
                "human_handoff_gate",
            ],
            "score": round(score, 3),
            "final_action": action,
            "rationale": rationale,
        })
        decisions.append({
            "candidate_id": row["candidate_id"],
            "score": score,
            "agent_action": action,
            "reference_action": row["reference_action"],
            "rationale": rationale,
        })
    return action_log, decisions


action_log, decisions = run_agentic_triage(rows)
print("Agentic workflow decisions:")
for item in decisions:
    print(item["candidate_id"], round(item["score"], 3), item["agent_action"], item["reference_action"], item["rationale"])


In [ ]:
def confusion(decision_rows):
    matrix = {}
    for item in decision_rows:
        truth = item["reference_action"]
        predicted = item["agent_action"]
        matrix.setdefault(truth, {})
        matrix[truth][predicted] = matrix[truth].get(predicted, 0) + 1
    return matrix


def accuracy(decision_rows):
    return sum(item["agent_action"] == item["reference_action"] for item in decision_rows) / len(decision_rows)


agent_follow_now = [item for item in decisions if item["agent_action"] == "follow_now"]
agent_precision = sum(item["reference_action"] == "follow_now" for item in agent_follow_now) / len(agent_follow_now)
manual_review_ids = [item["candidate_id"] for item in decisions if item["agent_action"] == "manual_review"]

print("agent confusion matrix:", confusion(decisions))
print("agent agreement with reference:", round(accuracy(decisions), 3))
print("agent follow-now ids:", [item["candidate_id"] for item in agent_follow_now])
print("agent follow-now precision:", round(agent_precision, 3))
print("manual-review queue:", manual_review_ids)


In [ ]:
print("Representative action logs:")
for candidate_id in ["T03", "T06", "T12"]:
    item = next(entry for entry in action_log if entry["candidate_id"] == candidate_id)
    print()
    print(candidate_id)
    print("tool sequence =", item["tool_sequence"])
    print("score =", item["score"])
    print("final action =", item["final_action"])
    print("rationale =", item["rationale"])

print()
print("Interpretation:")
print("  T03 shows why a high alert score alone is not enough: the artifact gate must run before ranking.")
print("  T06 shows why agents need a human-handoff state instead of forcing every case into follow or reject.")
print("  T12 shows the happy path: once the gates pass and the score is high, the workflow can stop early.")


In [ ]:
assistant_planning_prompt = """你是我的科研工作流助手。

任务：
1. 先阅读候选体表的列定义；
2. 明确哪些工具步骤必须先于排序执行；
3. 给出 stop conditions 和 human handoff 条件；
4. 最后输出一个 follow-now 队列和一个 manual-review 队列。

候选体表字段：
- candidate_id
- alert_score
- rise_rate_mag_day
- color_g_r
- crossmatch_arcsec
- image_quality
- moving_object_flag

科学与流程约束：
- artifact 和 moving_object 不能直接进入高优先级列表；
- 与已知源过近的目标不应被误当作新瞬变；
- 边界样本要进入人工复核，而不是被代理强行拍板；
- 排序之前必须先跑质量门和 novelty gate。

输出格式：
- Required tools
- Proposed order
- Stop conditions
- Human handoff rules
- Final queues
"""

print(assistant_planning_prompt)


In [ ]:
workflow_checklist = [
    "先定义工具边界：哪些函数能读数据、打分、过滤和生成摘要。",
    "让代理在每一步留下 action log，而不是只给最终答案。",
    "把 reject、follow_now、manual_review 三种出口写清楚。",
    "任何边界样本都要允许停在 human handoff，而不是被迫自动决策。",
    "在真实项目里，把代理日志和最终人工决定一起保存进实验记录。",
]

print("Agentic workflow checklist:")
for index, item in enumerate(workflow_checklist, start=1):
    print(f"{index}. {item}")

snapshot = {
    "candidate_count": len(rows),
    "baseline_precision_at_3": round(baseline_precision_at_3, 3),
    "agent_agreement": round(accuracy(decisions), 3),
    "follow_now_count": len(agent_follow_now),
    "manual_review_count": len(manual_review_ids),
}

print("Workflow snapshot:")
for key, value in snapshot.items():
    print(f"  {key}: {value}")


In [ ]:
try:
    import matplotlib.pyplot as plt

    ordered = sorted(decisions, key=lambda item: item["score"], reverse=True)
    labels = [item["candidate_id"] for item in ordered]
    scores = [item["score"] for item in ordered]
    colors = {
        "follow_now": "#1f7a4d",
        "manual_review": "#c97a00",
        "reject": "#b03a2e",
    }

    plt.figure(figsize=(8.0, 3.8))
    plt.bar(labels, scores, color=[colors[item["agent_action"]] for item in ordered])
    plt.axhline(0.78, color="#1f7a4d", linestyle="--", label="follow threshold")
    plt.axhline(0.58, color="#c97a00", linestyle=":", label="review threshold")
    plt.ylabel("follow-up score")
    plt.title("Agentic triage on the tiny transient table")
    plt.grid(axis="y", alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f"Optional matplotlib plot skipped: {exc}")


In [ ]:
from part5_delivery_exports import export_ch36_agentic_workflow_delivery

export_ch36_agentic_workflow_delivery(globals())
